
# Notebook 05 — Kiểm định p-value ghép cặp theo 577 bệnh nhân
## XGBoost + GLCM + kernelRadius=5: `none` vs `balanced`

Notebook này:
- đối chiếu `none` và `balanced` trên cùng 577 bệnh nhân;
- xác minh cùng validation fold và cùng OOF sample trước khi ghép cặp;
- tính metric ở mức voxel riêng cho từng bệnh nhân;
- chạy Wilcoxon signed-rank hai phía;
- tính rank-biserial correlation làm effect size;
- hiệu chỉnh nhiều phép kiểm định bằng Holm;
- xuất bảng và đoạn mô tả sẵn để chèn vào Chương 4.

**Đơn vị kiểm định là bệnh nhân (n=577), không phải từng voxel độc lập.**
Các metric của mỗi bệnh nhân vẫn là metric phân loại voxel trong bệnh nhân đó, không phải chẩn đoán ở cấp bệnh nhân.

Các metric phụ thuộc ngưỡng dùng `T=0.50`, đúng với thí nghiệm class-weight ablation.


In [ ]:

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon, rankdata
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix,
    roc_auc_score, average_precision_score
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

THRESHOLD = 0.50
ALPHA = 0.05
EXPECTED_PATIENTS = 577
EXPECTED_ROWS = 653_397

METRICS = [
    "accuracy", "balanced_accuracy", "precision", "recall", "specificity",
    "f1", "mcc", "roc_auc", "pr_auc"
]
PRIMARY_METRICS = ["f1", "roc_auc", "pr_auc"]

OUTPUT_DIR = Path("output/577p/glcm/kernel5/pvalue_patient_level")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ABLATION_DIR = Path("output/577p/glcm/kernel5/class_weight_ablation")
NONE_PATH = ABLATION_DIR / "development_oof_predictions_none.csv"
BALANCED_PATH = ABLATION_DIR / "development_oof_predictions_balanced.csv"
COMBINED_PATH = ABLATION_DIR / "class_weight_ablation_oof_predictions.csv"
OFFICIAL_BALANCED_PATH = ABLATION_DIR / "development_oof_predictions.csv"

print("OUTPUT_DIR:", OUTPUT_DIR)



## 1. Hàm hỗ trợ

Effect size dùng **rank-biserial correlation** với:
`delta = Balanced - None`.

- `r_rb > 0`: balanced cao hơn;
- `r_rb < 0`: none cao hơn.


In [ ]:

def binary_metrics(y_true, prob, threshold=0.50):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob).astype(float)
    pred = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    out = {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred),
    }

    if np.unique(y_true).size == 2:
        out["roc_auc"] = roc_auc_score(y_true, prob)
        out["pr_auc"] = average_precision_score(y_true, prob)
    else:
        out["roc_auc"] = np.nan
        out["pr_auc"] = np.nan

    return out


def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adjusted = np.empty(m, dtype=float)
    running_max = 0.0

    for k, idx in enumerate(order):
        candidate = (m - k) * p[idx]
        running_max = max(running_max, candidate)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


def rank_biserial_signed(diff):
    d = np.asarray(diff, dtype=float)
    d = d[np.isfinite(d)]
    d = d[d != 0]

    if len(d) == 0:
        return 0.0, 0.0, 0.0, 0

    ranks = rankdata(np.abs(d), method="average")
    w_plus = float(ranks[d > 0].sum())
    w_minus = float(ranks[d < 0].sum())
    denom = w_plus + w_minus
    r_rb = (w_plus - w_minus) / denom if denom else 0.0
    return float(r_rb), w_plus, w_minus, int(len(d))


def effect_label(r):
    a = abs(float(r))
    if a < 0.10:
        return "Không đáng kể"
    if a < 0.30:
        return "Nhỏ"
    if a < 0.50:
        return "Trung bình"
    return "Lớn"


## 2. Nạp OOF `none` và `balanced`

In [ ]:

def load_oof_pair():
    if NONE_PATH.exists() and BALANCED_PATH.exists():
        return pd.read_csv(NONE_PATH), pd.read_csv(BALANCED_PATH), "separate condition files"

    if COMBINED_PATH.exists():
        all_oof = pd.read_csv(COMBINED_PATH)
        if "condition" not in all_oof.columns:
            raise ValueError(f"{COMBINED_PATH} thiếu cột condition.")
        none = all_oof.loc[all_oof["condition"].astype(str).eq("none")].copy()
        balanced = all_oof.loc[all_oof["condition"].astype(str).eq("balanced")].copy()
        return none, balanced, f"combined file: {COMBINED_PATH}"

    balanced_candidates = [
        BALANCED_PATH,
        OFFICIAL_BALANCED_PATH,
        Path("development_oof_predictions.csv"),
        Path("./output/577p/glcm/kernel5/class_weight_ablation/development_oof_predictions.csv"),
    ]
    found_bal = next((p for p in balanced_candidates if p.exists()), None)

    msg = [
        "Không tìm thấy đủ OOF cho cả none và balanced.",
        f"Đã kiểm tra: {NONE_PATH}",
        f"Đã kiểm tra: {BALANCED_PATH}",
        f"Đã kiểm tra: {COMBINED_PATH}",
    ]
    if found_bal is not None:
        msg += [f"Đã tìm thấy balanced OOF tại: {found_bal}", "Thiếu OOF condition='none'."]
    msg += [
        "Hãy chạy Notebook 01 để tạo development_oof_predictions_none.csv và "
        "development_oof_predictions_balanced.csv (hoặc class_weight_ablation_oof_predictions.csv), "
        "sau đó chạy lại Notebook 05."
    ]
    raise FileNotFoundError("\n".join(msg))


none, balanced, oof_source = load_oof_pair()
print("OOF source:", oof_source)
print("none:", none.shape)
print("balanced:", balanced.shape)



## 3. Audit ghép cặp

Notebook không tự suy đoán pairing. Trước khi tính p-value, nó yêu cầu:
1. cùng số dòng;
2. cùng tập patient_id;
3. cùng nhãn thật;
4. cùng fold;
5. nếu có tọa độ voxel thì dùng tọa độ để ghép;
6. nếu không có tọa độ, chuỗi `patient_id + label + fold` phải trùng row-by-row.


In [ ]:

REQUIRED = {"patient_id", "label", "fold", "prob_tumor"}

for name, df in [("none", none), ("balanced", balanced)]:
    missing = REQUIRED - set(df.columns)
    if missing:
        raise ValueError(f"{name} OOF thiếu cột: {sorted(missing)}")

    df["patient_id"] = df["patient_id"].astype(str)
    df["label"] = df["label"].astype(int)
    df["fold"] = df["fold"].astype(int)
    df["prob_tumor"] = df["prob_tumor"].astype(float)

if len(none) != len(balanced):
    raise RuntimeError(f"Row count khác nhau: none={len(none)}, balanced={len(balanced)}")

none_patients = set(none["patient_id"].unique())
bal_patients = set(balanced["patient_id"].unique())
if none_patients != bal_patients:
    raise RuntimeError(
        f"Patient mismatch: missing_in_balanced={len(none_patients-bal_patients)}, "
        f"extra_in_balanced={len(bal_patients-none_patients)}"
    )

coord_candidates = [
    ["x", "y", "z"],
    ["i", "j", "k"],
    ["voxel_x", "voxel_y", "voxel_z"],
]
coords = next(
    (cols for cols in coord_candidates
     if set(cols).issubset(none.columns) and set(cols).issubset(balanced.columns)),
    None
)

if coords:
    key = ["patient_id", "fold", "label"] + coords
    if none.duplicated(key).any() or balanced.duplicated(key).any():
        raise RuntimeError(f"Pairing key có duplicate: {key}")

    paired = none[key + ["prob_tumor"]].rename(columns={"prob_tumor": "prob_none"}).merge(
        balanced[key + ["prob_tumor"]].rename(columns={"prob_tumor": "prob_balanced"}),
        on=key, how="inner", validate="one_to_one"
    )
    pairing_strategy = "patient_id + fold + label + voxel coordinates"
else:
    compare_cols = ["patient_id", "label", "fold"]
    left = none[compare_cols].reset_index(drop=True)
    right = balanced[compare_cols].reset_index(drop=True)

    if not left.equals(right):
        raise RuntimeError(
            "Không có tọa độ voxel và thứ tự patient_id/label/fold không trùng row-by-row. "
            "Không thể chứng minh cùng sample voxel nên notebook dừng."
        )

    paired = left.copy()
    paired["prob_none"] = none["prob_tumor"].to_numpy()
    paired["prob_balanced"] = balanced["prob_tumor"].to_numpy()
    pairing_strategy = "row-wise identity of patient_id + label + fold"

folds_per_patient = paired.groupby("patient_id")["fold"].nunique()
if not (folds_per_patient == 1).all():
    bad = folds_per_patient[folds_per_patient != 1]
    raise RuntimeError(f"Có patient xuất hiện ở nhiều validation fold: {bad.head().to_dict()}")

audit = {
    "oof_source": oof_source,
    "pairing_strategy": pairing_strategy,
    "paired_rows": int(len(paired)),
    "paired_patients": int(paired["patient_id"].nunique()),
    "folds": sorted(map(int, paired["fold"].unique())),
    "same_label_and_fold_verified": True,
}

print(json.dumps(audit, indent=2, ensure_ascii=False))

if paired["patient_id"].nunique() != EXPECTED_PATIENTS:
    warnings.warn(
        f"Số bệnh nhân = {paired['patient_id'].nunique()}, khác EXPECTED_PATIENTS={EXPECTED_PATIENTS}."
    )
if len(paired) != EXPECTED_ROWS:
    warnings.warn(f"Số voxel = {len(paired)}, khác EXPECTED_ROWS={EXPECTED_ROWS}.")

with open(OUTPUT_DIR / "05_pairing_audit.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)


## 4. Tính metric riêng cho từng bệnh nhân

In [ ]:

patient_rows = []

for patient_id, g in paired.groupby("patient_id", sort=True):
    y = g["label"].to_numpy()

    m_none = binary_metrics(y, g["prob_none"].to_numpy(), threshold=THRESHOLD)
    m_bal = binary_metrics(y, g["prob_balanced"].to_numpy(), threshold=THRESHOLD)

    row = {
        "patient_id": patient_id,
        "fold": int(g["fold"].iloc[0]),
        "n_voxels": int(len(g)),
        "n_non_tumor": int((y == 0).sum()),
        "n_tumor": int((y == 1).sum()),
    }

    for metric in METRICS:
        row[f"none_{metric}"] = m_none[metric]
        row[f"balanced_{metric}"] = m_bal[metric]
        row[f"delta_balanced_minus_none_{metric}"] = m_bal[metric] - m_none[metric]

    patient_rows.append(row)

patient_metrics = pd.DataFrame(patient_rows).sort_values("patient_id").reset_index(drop=True)
patient_metrics.to_csv(OUTPUT_DIR / "05_per_patient_metrics_none_balanced.csv", index=False)

print("Patients:", len(patient_metrics))
print(
    "Patients with both classes:",
    ((patient_metrics["n_non_tumor"] > 0) & (patient_metrics["n_tumor"] > 0)).sum()
)
display(patient_metrics.head())



## 5. Wilcoxon + effect size + Holm

- Wilcoxon signed-rank hai phía;
- `zero_method="wilcox"`;
- Holm chính trên 9 metric;
- xuất thêm Holm cho 3 metric chính: F1, ROC-AUC, PR-AUC.


In [ ]:

stat_rows = []

for metric in METRICS:
    a = patient_metrics[f"none_{metric}"].astype(float)
    b = patient_metrics[f"balanced_{metric}"].astype(float)

    valid = a.notna() & b.notna()
    a = a[valid].to_numpy()
    b = b[valid].to_numpy()
    d = b - a

    nonzero = d[d != 0]
    if len(nonzero) == 0:
        W = 0.0
        p_raw = 1.0
    else:
        test = wilcoxon(
            b, a,
            alternative="two-sided",
            zero_method="wilcox",
            correction=False,
            method="auto",
        )
        W = float(test.statistic)
        p_raw = float(test.pvalue)

    r_rb, w_plus, w_minus, n_nonzero = rank_biserial_signed(d)

    stat_rows.append({
        "metric": metric,
        "n_paired": int(len(d)),
        "n_nonzero_diff": int(n_nonzero),
        "none_mean": float(np.nanmean(a)),
        "none_sd": float(np.nanstd(a, ddof=1)),
        "balanced_mean": float(np.nanmean(b)),
        "balanced_sd": float(np.nanstd(b, ddof=1)),
        "none_median": float(np.nanmedian(a)),
        "balanced_median": float(np.nanmedian(b)),
        "median_delta_balanced_minus_none": float(np.nanmedian(d)),
        "mean_delta_balanced_minus_none": float(np.nanmean(d)),
        "wilcoxon_W": W,
        "p_raw": p_raw,
        "W_plus": w_plus,
        "W_minus": w_minus,
        "rank_biserial_r": r_rb,
        "effect_magnitude": effect_label(r_rb),
        "direction": (
            "balanced > none" if r_rb > 0 else
            "balanced < none" if r_rb < 0 else
            "không khác theo hướng"
        ),
        "patients_balanced_better": int((d > 0).sum()),
        "patients_none_better": int((d < 0).sum()),
        "patients_tie": int((d == 0).sum()),
    })

stats = pd.DataFrame(stat_rows)
stats["p_holm_all9"] = holm_adjust(stats["p_raw"].to_numpy())
stats["significant_holm_all9"] = stats["p_holm_all9"] < ALPHA

stats["p_holm_primary3"] = np.nan
mask_primary = stats["metric"].isin(PRIMARY_METRICS)
stats.loc[mask_primary, "p_holm_primary3"] = holm_adjust(
    stats.loc[mask_primary, "p_raw"].to_numpy()
)

stats.to_csv(OUTPUT_DIR / "05_wilcoxon_holm_all_metrics.csv", index=False)
display(stats)


## 6. Bảng sẵn để chèn vào Chương 4

In [ ]:

VI_NAMES = {
    "accuracy": "Accuracy",
    "balanced_accuracy": "Balanced Accuracy",
    "precision": "Precision",
    "recall": "Recall",
    "specificity": "Specificity",
    "f1": "F1-score",
    "mcc": "MCC",
    "roc_auc": "ROC-AUC",
    "pr_auc": "PR-AUC",
}

rows = []
for _, r in stats.iterrows():
    rows.append({
        "Chỉ số": VI_NAMES[r["metric"]],
        "Không trọng số (TB ± SD)": f"{r['none_mean']:.4f} ± {r['none_sd']:.4f}",
        "Balanced (TB ± SD)": f"{r['balanced_mean']:.4f} ± {r['balanced_sd']:.4f}",
        "Δ trung vị (Balanced − None)": f"{r['median_delta_balanced_minus_none']:.4f}",
        "p Wilcoxon": f"{r['p_raw']:.6g}",
        "p Holm (9 chỉ số)": f"{r['p_holm_all9']:.6g}",
        "r_rb": f"{r['rank_biserial_r']:.4f}",
        "Mức ảnh hưởng": r["effect_magnitude"],
        "Kết luận α=0,05": "Có ý nghĩa" if r["significant_holm_all9"] else "Không có ý nghĩa",
    })

chapter4 = pd.DataFrame(rows)
chapter4.to_csv(
    OUTPUT_DIR / "05_chapter4_table_4_12a.csv",
    index=False,
    encoding="utf-8-sig"
)

md_lines = [
    "**Bảng 4.12a. Kiểm định Wilcoxon signed-rank ghép cặp theo 577 bệnh nhân đối với ảnh hưởng của trọng số lớp**",
    "",
    "Ghi chú: metric được tính ở mức voxel riêng cho từng bệnh nhân từ OOF; "
    "Δ = Balanced − None; p-value được hiệu chỉnh Holm trên 9 chỉ số; "
    "r_rb là rank-biserial correlation.",
    "",
    chapter4.to_markdown(index=False),
]
(OUTPUT_DIR / "05_chapter4_table_4_12a.md").write_text(
    "\n".join(md_lines), encoding="utf-8"
)

display(chapter4)


## 7. Đoạn mô tả sẵn cho Chương 4

In [ ]:

sig = stats.loc[stats["significant_holm_all9"]].copy()
not_sig = stats.loc[~stats["significant_holm_all9"]].copy()

def metric_vi(m):
    return VI_NAMES.get(m, m)

parts = [
    "Để đánh giá ảnh hưởng của cơ chế gán trọng số lớp bằng kiểm định thống kê ghép cặp, "
    "các dự đoán ngoài phân hoạch của hai điều kiện không dùng trọng số và dùng trọng số cân bằng "
    "được đối chiếu trên cùng 577 bệnh nhân và cùng phân hoạch StratifiedGroupKFold. "
    "Đối với mỗi bệnh nhân, các chỉ số phân loại voxel được tính riêng ở ngưỡng 0,50, sau đó "
    "so sánh bằng kiểm định Wilcoxon signed-rank hai phía. "
    "Do nhiều chỉ số được kiểm định đồng thời, p-value được hiệu chỉnh theo phương pháp Holm."
]

if len(sig):
    desc = []
    for _, r in sig.iterrows():
        direction = (
            "cao hơn với trọng số cân bằng"
            if r["rank_biserial_r"] > 0
            else "cao hơn khi không dùng trọng số"
        )
        desc.append(
            f"{metric_vi(r['metric'])} "
            f"(p_Holm={r['p_holm_all9']:.4g}, r_rb={r['rank_biserial_r']:.3f}; {direction})"
        )
    parts.append(
        "Sau hiệu chỉnh Holm ở mức α=0,05, các chỉ số có khác biệt có ý nghĩa thống kê gồm: "
        + "; ".join(desc) + "."
    )
else:
    parts.append(
        "Sau hiệu chỉnh Holm ở mức α=0,05, chưa có chỉ số nào đạt ngưỡng ý nghĩa thống kê."
    )

if len(not_sig):
    parts.append(
        "Các chỉ số còn lại không được kết luận là khác biệt có ý nghĩa thống kê sau hiệu chỉnh nhiều phép kiểm định."
    )

parts.append(
    "Kết quả này chỉ phản ánh thay đổi hiệu năng phân loại voxel giữa hai cơ chế trọng số trên cùng bệnh nhân; "
    "không được diễn giải là hiệu năng chẩn đoán ở cấp bệnh nhân."
)

text_ready = "\n\n".join(parts)
(OUTPUT_DIR / "05_chapter4_text_ready.txt").write_text(text_ready, encoding="utf-8")
print(text_ready)



## 8. File đầu ra

Sau khi chạy thành công:

- `05_pairing_audit.json`
- `05_per_patient_metrics_none_balanced.csv`
- `05_wilcoxon_holm_all_metrics.csv`
- `05_chapter4_table_4_12a.csv`
- `05_chapter4_table_4_12a.md`
- `05_chapter4_text_ready.txt`

Notebook này **không sử dụng Hold-out Validation Cohort**.
